# ATLAS Wind Atlas: conversion of future scenario components

This notebook converts the future corrected wind components produced by the **wind delta workflow** into wind speed and wind direction.

It reads, for each selected month and area:

1. the future corrected **u** component
2. the future corrected **v** component
3. the present day integrated wind dataset produced by the wind conversion workflow

It then creates one final NetCDF dataset with 8 variables:

```text
u10_corrected
v10_corrected
wds_corrected
dir_corrected
delta_u10
delta_v10
delta_wds
delta_dir
```

The notebook is designed to process the continental area first and the islands afterwards, using the same methodology for both areas.


## Method overview

The wind delta notebook computes the future corrected wind components as:

```text
future corrected component = present day integrated component + climate change delta
```

This notebook starts from the corrected `u` and `v` components and computes:

```text
wind speed = sqrt(u² + v²)
```

Wind direction follows the meteorological convention:

```text
0°   = wind coming from North
90°  = wind coming from East
180° = wind coming from South
270° = wind coming from West
```

The wind speed delta is computed as the difference between future corrected wind speed and present day integrated wind speed.

The wind direction delta is computed using a circular difference so that the result stays in the range `-180° to +180°`. This avoids artificial jumps around 0° and 360°.


## 0. Load libraries

Run this cell first. It imports the Python packages used to open NetCDF files, compute wind variables and save the final datasets.


In [1]:
from pathlib import Path
import gc
import warnings

import numpy as np
import rioxarray  # noqa: F401. Needed to activate the .rio accessor in xarray
import xarray as xr
from dask.diagnostics import ProgressBar

warnings.filterwarnings("ignore")

## 1. User settings

Edit only this cell for a standard run.

The notebook processes `continental` first and `islands` afterwards by default. If one of the two areas is not available, remove it from `AREA_NAMES`.


In [2]:
# Country or study area name used in folder names and file names.
COUNTRY = "chile"

# CMIP6 model and scenario used in the previous scenario notebooks.
MODEL = "CNRM-ESM2-1"
EXPERIMENT = "ssp370"

# Future period used in the previous delta notebook.
START_YEAR = 2020
END_YEAR = 2050

# Months to process. Use list(range(1, 13)) for the full year.
MONTHS = list(range(1, 2))
MONTH = 1
# Areas to process. The order matters only for logging.
# Use ["continental"] or ["islands"] if only one area is available.
AREA_NAMES = ["continental", "islands"]
AREA_NAME = 'islands'
# Variable names expected in the corrected component files produced by the delta notebook.
U_CORRECTED_VAR = "u10_corrected"
V_CORRECTED_VAR = "v10_corrected"
DELTA_VAR = "delta"

# Variable names expected in the present day integrated wind file produced by the conversion notebook.
PRESENT_WDS_VAR = "wds_integrated"
PRESENT_DIR_VAR = "dir_integrated"

# Output variable names.
WDS_CORRECTED_VAR = "wds_corrected"
DIR_CORRECTED_VAR = "dir_corrected"
DELTA_U_VAR = "delta_u10"
DELTA_V_VAR = "delta_v10"
DELTA_WDS_VAR = "delta_wds"
DELTA_DIR_VAR = "delta_dir"

## 2. Folder configuration

The paths are relative and anonymous, so the notebook can be shared without exposing local server folders.

Expected inputs from the delta notebook:

```text
../data/atlas_data/<country>/<model>/<scenario>/sub_areas/u10_corrected_...
../data/atlas_data/<country>/<model>/<scenario>/sub_areas/v10_corrected_...
```

Expected present day integrated input from the wind conversion notebook:

```text
../data/atlas_data/<country>/10m_wind_integrated_...
```


In [3]:
# Base project folder.
BASE_DIR = Path("../data")

# Main ATLAS folder for the selected country.
ATLAS_DIR = BASE_DIR / "atlas_data" / COUNTRY

# Folder containing the corrected u and v component files produced by the wind delta notebook.
CORRECTED_COMPONENT_DIR = ATLAS_DIR / MODEL / EXPERIMENT / "sub_areas"

# Folder containing the present day integrated wind files produced by the previous conversion notebook.
PRESENT_WIND_DIR = ATLAS_DIR

# Folder where the final scenario wind datasets will be written.
OUTPUT_DIR = ATLAS_DIR / MODEL / EXPERIMENT
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Corrected component folder:", CORRECTED_COMPONENT_DIR)
print("Present day wind folder:", PRESENT_WIND_DIR)
print("Output folder:", OUTPUT_DIR)

Corrected component folder: ../data/atlas_data/chile/CNRM-ESM2-1/ssp370/sub_areas
Present day wind folder: ../data/atlas_data/chile
Output folder: ../data/atlas_data/chile/CNRM-ESM2-1/ssp370


## 3. Helper functions

These functions keep the workflow readable and make error messages clearer for non expert users.


In [4]:
def check_required_file(path):
    """Stop the notebook with a clear message if a required file is missing."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Required input file not found: {path}\\n"
            "Check COUNTRY, MODEL, EXPERIMENT, AREA_NAMES, START_YEAR, END_YEAR and the folder structure."
        )
    return path


def drop_spatial_ref(ds):
    """Remove the auxiliary spatial reference variable when present."""
    return ds.drop_vars("spatial_ref", errors="ignore")


def get_spatial_coordinate_names(ds):
    """Return the latitude and longitude coordinate names used by a dataset."""
    lat_candidates = ["latitude", "lat", "y"]
    lon_candidates = ["longitude", "lon", "x"]

    lat_name = next((name for name in lat_candidates if name in ds.coords or name in ds.dims), None)
    lon_name = next((name for name in lon_candidates if name in ds.coords or name in ds.dims), None)

    if lat_name is None or lon_name is None:
        raise ValueError(
            "Could not identify latitude and longitude coordinates. "
            f"Available coordinates are: {list(ds.coords)}"
        )

    return lat_name, lon_name


def get_default_chunks(path):
    """Return a safe chunk dictionary based on the dimensions present in a NetCDF file."""
    with xr.open_dataset(path) as ds:
        chunks = {}
        for dim in ds.dims:
            if dim in ["latitude", "lat", "y"]:
                chunks[dim] = 1000
            elif dim in ["longitude", "lon", "x"]:
                chunks[dim] = 1000
            else:
                chunks[dim] = 1
    return chunks


def open_dataset_checked(path):
    """Open a NetCDF file after checking that it exists."""
    path = check_required_file(path)
    chunks = get_default_chunks(path)
    ds = xr.open_dataset(path, chunks=chunks)
    ds = ds.rio.write_crs("EPSG:4326")
    ds = drop_spatial_ref(ds)
    return ds


def sort_spatial_coordinates(ds):
    """Sort the dataset by latitude and longitude when these coordinates are present."""
    lat_name, lon_name = get_spatial_coordinate_names(ds)

    if lat_name in ds.coords:
        ds = ds.sortby(lat_name)
    if lon_name in ds.coords:
        ds = ds.sortby(lon_name)

    return ds


def get_first_available_variable(ds, candidates):
    """Return the first variable name that exists in a dataset."""
    for candidate in candidates:
        if candidate in ds.data_vars:
            return candidate

    raise KeyError(
        "None of the expected variables were found. "
        f"Expected one of: {candidates}. "
        f"Available variables are: {list(ds.data_vars)}"
    )


def save_xarray_netcdf_fast(ds, output_path):
    """Save an xarray Dataset as compressed NetCDF using conservative chunk sizes."""
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    ds = ds.astype("float32")

    encoding = {}
    for var in ds.data_vars:
        dims = ds[var].dims
        shape = ds[var].shape
        chunksizes = []

        for dim, size in zip(dims, shape):
            if dim in ["latitude", "lat", "y"]:
                chunksizes.append(min(2048, size))
            elif dim in ["longitude", "lon", "x"]:
                chunksizes.append(min(2048, size))
            else:
                chunksizes.append(min(1, size))

        encoding[var] = {
            "dtype": "float32",
            "zlib": True,
            "complevel": 1,
            "shuffle": True,
            "chunksizes": tuple(chunksizes),
        }

    delayed = ds.to_netcdf(
        output_path,
        engine="h5netcdf",
        encoding=encoding,
        compute=False,
        mode="w",
    )

    with ProgressBar():
        delayed.compute(scheduler="single-threaded")

## 4. File name helpers

If previous notebooks use a different naming convention, update only these functions.


In [5]:
def corrected_component_file(component, month, area_name):
    """Return the path to a corrected u or v component file produced by the delta notebook."""
    file_name = (
        f"{component}10_corrected_"
        f"{COUNTRY}_m{month}_{EXPERIMENT}_{area_name}_{START_YEAR}_{END_YEAR}.nc"
    )
    return CORRECTED_COMPONENT_DIR / file_name


def present_wind_file(month, area_name):
    """Return the path to the present day integrated wind file produced by the conversion notebook."""
    file_name = f"10m_wind_integrated_{COUNTRY}_m{month}_{area_name}.nc"
    return PRESENT_WIND_DIR / file_name


def scenario_wind_output_file(month, area_name):
    """Return the path to the final scenario wind dataset."""
    file_name = (
        f"10m_wind_corrected_"
        f"{COUNTRY}_m{month}_{EXPERIMENT}_{area_name}_{START_YEAR}_{END_YEAR}.nc"
    )
    return OUTPUT_DIR / file_name

## 5. Wind calculation functions

These functions calculate wind speed, wind direction and their deltas.


In [6]:
def calculate_wind_speed(u, v):
    """Calculate wind speed from u and v components."""
    return np.sqrt(u**2 + v**2)


def calculate_wind_direction(u, v):
    """Calculate meteorological wind direction from u and v components.

    The result is expressed in degrees from 0 to 360.
    Direction indicates where the wind comes from.
    """
    return (np.rad2deg(np.arctan2(-u, -v)) + 360) % 360


def circular_direction_difference(future_direction, present_direction):
    """Calculate the shortest angular difference between two directions.

    The result is in the range -180 to +180 degrees.
    """
    return ((future_direction - present_direction + 180) % 360) - 180


def add_variable_metadata(ds):
    """Add simple metadata to the final output variables."""
    metadata = {
        U_CORRECTED_VAR: {
            "long_name": "Future corrected 10 m eastward wind component",
            "units": "m s-1",
        },
        V_CORRECTED_VAR: {
            "long_name": "Future corrected 10 m northward wind component",
            "units": "m s-1",
        },
        WDS_CORRECTED_VAR: {
            "long_name": "Future corrected 10 m wind speed",
            "units": "m s-1",
        },
        DIR_CORRECTED_VAR: {
            "long_name": "Future corrected 10 m meteorological wind direction",
            "units": "degrees",
            "description": "Direction indicates where wind comes from.",
        },
        DELTA_U_VAR: {
            "long_name": "Projected change in 10 m eastward wind component",
            "units": "m s-1",
        },
        DELTA_V_VAR: {
            "long_name": "Projected change in 10 m northward wind component",
            "units": "m s-1",
        },
        DELTA_WDS_VAR: {
            "long_name": "Projected change in 10 m wind speed",
            "units": "m s-1",
        },
        DELTA_DIR_VAR: {
            "long_name": "Projected change in 10 m wind direction",
            "units": "degrees",
            "description": "Circular difference in the range -180 to +180 degrees.",
        },
    }

    for var, attrs in metadata.items():
        if var in ds:
            ds[var].attrs.update(attrs)

    ds.attrs.update({
        "method": "Future corrected u and v components converted to wind speed and direction; deltas computed against present day integrated wind dataset.",
        "experiment": EXPERIMENT,
        "model": MODEL,
        "future_period": f"{START_YEAR} to {END_YEAR}",
    })

    return ds

## 6. Convert one month and one area

This function is the core of the notebook. It reads the corrected `u` and `v` files, calculates wind speed and direction, opens the present day integrated wind dataset, and computes the deltas for wind speed and direction.


In [7]:
def convert_scenario_month_area(month, area_name):
    """Create the final 8 variable scenario wind dataset for one month and one area."""
    print("=" * 80)
    print(f"Processing area: {area_name} | month: {month:02d}")

    u_path = corrected_component_file("u", month, area_name)
    v_path = corrected_component_file("v", month, area_name)
    present_path = present_wind_file(month, area_name)
    output_path = scenario_wind_output_file(month, area_name)

    print("Opening corrected u component:", u_path)
    ds_u = open_dataset_checked(u_path)

    print("Opening corrected v component:", v_path)
    ds_v = open_dataset_checked(v_path)

    print("Opening present day integrated wind dataset:", present_path)
    ds_present = open_dataset_checked(present_path)

    ds_u = sort_spatial_coordinates(ds_u)
    ds_v = sort_spatial_coordinates(ds_v)
    ds_present = sort_spatial_coordinates(ds_present)

    u_var = get_first_available_variable(ds_u, [U_CORRECTED_VAR, "u10", "uas", "uas_corrected"])
    v_var = get_first_available_variable(ds_v, [V_CORRECTED_VAR, "v10", "vas", "vas_corrected"])
    present_wds_var = get_first_available_variable(ds_present, [PRESENT_WDS_VAR, "wds", "wds_corrected"])
    present_dir_var = get_first_available_variable(ds_present, [PRESENT_DIR_VAR, "dir", "dir_corrected"])

    print("Aligning corrected u and v components...")
    u_corrected, v_corrected = xr.align(ds_u[u_var], ds_v[v_var], join="inner")

    print("Calculating corrected wind speed and direction...")
    wds_corrected = calculate_wind_speed(u_corrected, v_corrected)
    dir_corrected = calculate_wind_direction(u_corrected, v_corrected)

    print("Reading component deltas from corrected component files...")
    delta_u_var = get_first_available_variable(ds_u, [DELTA_VAR, DELTA_U_VAR, "delta_u"])
    delta_v_var = get_first_available_variable(ds_v, [DELTA_VAR, DELTA_V_VAR, "delta_v"])
    delta_u, delta_v = xr.align(ds_u[delta_u_var], ds_v[delta_v_var], join="inner")

    print("Aligning with present day wind speed and direction...")
    wds_corrected, present_wds = xr.align(wds_corrected, ds_present[present_wds_var], join="inner")
    dir_corrected, present_dir = xr.align(dir_corrected, ds_present[present_dir_var], join="inner")

    print("Calculating wind speed and wind direction deltas...")
    delta_wds = wds_corrected - present_wds
    delta_dir = circular_direction_difference(dir_corrected, present_dir)

    print("Building final dataset with 8 variables...")
    u_corrected, v_corrected, delta_u, delta_v, wds_corrected, dir_corrected, delta_wds, delta_dir = xr.align(
        u_corrected,
        v_corrected,
        delta_u,
        delta_v,
        wds_corrected,
        dir_corrected,
        delta_wds,
        delta_dir,
        join="inner",
    )

    output_ds = xr.Dataset(
        {
            U_CORRECTED_VAR: u_corrected,
            V_CORRECTED_VAR: v_corrected,
            WDS_CORRECTED_VAR: wds_corrected,
            DIR_CORRECTED_VAR: dir_corrected,
            DELTA_U_VAR: delta_u,
            DELTA_V_VAR: delta_v,
            DELTA_WDS_VAR: delta_wds,
            DELTA_DIR_VAR: delta_dir,
        }
    )

    output_ds = output_ds.rio.write_crs("EPSG:4326")
    output_ds = drop_spatial_ref(output_ds)
    output_ds = add_variable_metadata(output_ds)

    print("Saving final scenario wind dataset:", output_path)
    save_xarray_netcdf_fast(output_ds, output_path)

    ds_u.close()
    ds_v.close()
    ds_present.close()

    del ds_u, ds_v, ds_present, output_ds
    gc.collect()

    return output_path

## 7. Run the workflow

This cell processes all requested areas and months. By default it runs `continental` first and `islands` afterwards.


In [8]:
generated_files = []

for area_name in AREA_NAMES:
    for month in MONTHS:
        output_file = convert_scenario_month_area(month, area_name)
        generated_files.append(output_file)

print("=" * 80)
print("Scenario wind conversion completed.")
print("Generated files:")
for output_file in generated_files:
    print(output_file)

Processing area: continental | month: 01
Opening corrected u component: ../data/atlas_data/chile/CNRM-ESM2-1/ssp370/sub_areas/u10_corrected_chile_m1_ssp370_continental_2020_2050.nc


ERROR 1: PROJ: proj_create_from_database: Open of /home/alessandrom/anaconda3/envs/bias_correction_conda/share/proj failed


Opening corrected v component: ../data/atlas_data/chile/CNRM-ESM2-1/ssp370/sub_areas/v10_corrected_chile_m1_ssp370_continental_2020_2050.nc
Opening present day integrated wind dataset: ../data/atlas_data/chile/10m_wind_integrated_chile_m1_continental.nc
Aligning corrected u and v components...
Calculating corrected wind speed and direction...
Reading component deltas from corrected component files...
Aligning with present day wind speed and direction...
Calculating wind speed and wind direction deltas...
Building final dataset with 8 variables...
Saving final scenario wind dataset: ../data/atlas_data/chile/CNRM-ESM2-1/ssp370/10m_wind_corrected_chile_m1_ssp370_continental_2020_2050.nc
[########################################] | 100% Completed | 26m 55s
Processing area: islands | month: 01
Opening corrected u component: ../data/atlas_data/chile/CNRM-ESM2-1/ssp370/sub_areas/u10_corrected_chile_m1_ssp370_islands_2020_2050.nc
Opening corrected v component: ../data/atlas_data/chile/CNRM-ESM

## 8. Quick check

This optional cell opens the first generated file and prints its structure. The output should contain exactly the 8 expected variables.


In [9]:
if generated_files:
    check_ds = xr.open_dataset(generated_files[0])
    print(check_ds)
    print("Variables:", list(check_ds.data_vars))
    check_ds.close()

<xarray.Dataset> Size: 16GB
Dimensions:        (latitude: 46848, longitude: 10554)
Coordinates:
  * latitude       (latitude) float32 187kB -56.54 -56.54 -56.54 ... -17.5 -17.5
  * longitude      (longitude) float32 42kB -75.72 -75.72 ... -66.93 -66.93
Data variables:
    u10_corrected  (latitude, longitude) float32 2GB ...
    v10_corrected  (latitude, longitude) float32 2GB ...
    wds_corrected  (latitude, longitude) float32 2GB ...
    dir_corrected  (latitude, longitude) float32 2GB ...
    delta_u10      (latitude, longitude) float32 2GB ...
    delta_v10      (latitude, longitude) float32 2GB ...
    delta_wds      (latitude, longitude) float32 2GB ...
    delta_dir      (latitude, longitude) float32 2GB ...
Attributes:
    method:         Future corrected u and v components converted to wind spe...
    experiment:     ssp370
    model:          CNRM-ESM2-1
    future_period:  2020 to 2050
Variables: ['u10_corrected', 'v10_corrected', 'wds_corrected', 'dir_corrected', 'delta_u10